In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv
/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/LZ46LTL6O2.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/13H1VTAZ42.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/G0DWU2GK1L.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/CVPHC0DT1P.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/90XJFQDL7G.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/3V8LZ56MOW.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/Y9EUV6TNP2.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/K76P42J0GK.jpg
/kaggle/input/competitions/industrial-surface-defect-detection/public/test/ZSS7THB1UM.jpg
/kaggle/input/comp

In [2]:
import pandas as pd
import numpy as np
import os

CSV_PATH = "/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv"
train = pd.read_csv(CSV_PATH)

DEFECTS = ["defect_a", "defect_b", "defect_c", "defect_d"]
IMAGE_ID = "0ZUU8LH57Y"
H, W = 128, 800

def decode_rle(rle, height=H, width=W):
    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(rle) or str(rle).strip() == "":
        return mask.reshape((height, width), order="F")

    values = np.asarray(str(rle).split(), dtype=np.int64)
    starts = values[0::2] - 1
    lengths = values[1::2]

    for start, length in zip(starts, lengths):
        mask[start:start + length] = 1

    return mask.reshape((height, width), order="F")

# Q2: total active pixels for the specified image across all four classes
image_rows = train[train["image_id"] == IMAGE_ID]

print("Q2 — rows found:", len(image_rows))
print(image_rows[["image_id", "defect_type", "mask_rle"]])

pixel_total = 0
for defect in DEFECTS:
    rows = image_rows[image_rows["defect_type"] == defect]
    if len(rows):
        pixel_total += int(decode_rle(rows.iloc[0]["mask_rle"]).sum())

print("\nQ2 answer — total active pixels:", pixel_total)

# Q3: number of unique images containing each defect
presence = (
    train.assign(
        present=train["mask_rle"].notna()
        & train["mask_rle"].astype(str).str.strip().ne("")
    )
    .pivot_table(
        index="image_id",
        columns="defect_type",
        values="present",
        aggfunc="max",
        fill_value=False
    )
    .reindex(columns=DEFECTS, fill_value=False)
    .astype(bool)
)

counts = presence.sum().sort_values(ascending=False)

print("\nQ3 — images containing each defect:")
print(counts)

print("\nMost common:", counts.index[0], int(counts.iloc[0]))
print("Rarest:", counts.index[-1], int(counts.iloc[-1]))

Q2 — rows found: 4
         image_id defect_type  \
40804  0ZUU8LH57Y    defect_a   
40805  0ZUU8LH57Y    defect_b   
40806  0ZUU8LH57Y    defect_c   
40807  0ZUU8LH57Y    defect_d   

                                                mask_rle  
40804  88169 11 88183 6 88294 24 88422 24 88550 24 88...  
40805                                                NaN  
40806  37642 1 37769 2 37897 3 38025 4 38153 5 38281 ...  
40807  79113 13 79239 16 79367 17 79495 18 79622 21 7...  

Q2 answer — total active pixels: 6514

Q3 — images containing each defect:
defect_type
defect_c    4100
defect_d    1707
defect_a    1081
defect_b     313
dtype: int64

Most common: defect_c 4100
Rarest: defect_b 313


In [3]:
# Q4: image-level defect co-occurrence matrix
cooccurrence = presence.astype(int).T @ presence.astype(int)

print("Q4 — co-occurrence matrix:")
display(cooccurrence)

co = cooccurrence.to_numpy().copy()
np.fill_diagonal(co, -1)

i, j = np.unravel_index(np.argmax(co), co.shape)

print(
    "\nQ4 answer:",
    f"{DEFECTS[i]} + {DEFECTS[j]},",
    int(co[i, j])
)

# Q5: Pearson correlation matrix for binary image-level presence
correlation = presence.astype(int).corr(method="pearson")

print("\nQ5 — Pearson correlation matrix:")
display(correlation)

corr_values = correlation.to_numpy().copy()
np.fill_diagonal(corr_values, -np.inf)

ci, cj = np.unravel_index(np.argmax(corr_values), corr_values.shape)

print(
    "\nHighest-correlation pair:",
    f"{DEFECTS[ci]} + {DEFECTS[cj]}"
)
print("Pearson correlation:", round(float(corr_values[ci, cj]), 2))
print("Q5 answer — rounded correlation:", f"{corr_values[ci, cj]:.2f}")

Q4 — co-occurrence matrix:


defect_type,defect_a,defect_b,defect_c,defect_d
defect_type,,,,
defect_a,1081,42,310,234
defect_b,42,313,95,64
defect_c,310,95,4100,705
defect_d,234,64,705,1707



Q4 answer: defect_c + defect_d, 705

Q5 — Pearson correlation matrix:


defect_type,defect_a,defect_b,defect_c,defect_d
defect_type,,,,
defect_a,1.000000,0.022541,-0.049311,0.060452
defect_b,0.022541,1.000000,-0.019735,0.025733
defect_c,-0.049311,-0.019735,1.000000,0.046475
defect_d,0.060452,0.025733,0.046475,1.000000



Highest-correlation pair: defect_a + defect_d
Pearson correlation: 0.06
Q5 answer — rounded correlation: 0.06


In [4]:
# Q6: inspect all images with more than one non-empty defect mask
image_ids = train["image_id"].drop_duplicates().tolist()

max_overlap_pixels = 0
max_overlap_details = None
any_overlap = False
max_dice = 0.0
max_dice_details = None

for image_id in image_ids:
    rows = train[train["image_id"] == image_id]
    masks = {}

    for defect in DEFECTS:
        match = rows[rows["defect_type"] == defect]
        rle = match.iloc[0]["mask_rle"] if len(match) else None
        masks[defect] = decode_rle(rle).astype(bool)

    for a in range(len(DEFECTS)):
        for b in range(a + 1, len(DEFECTS)):
            ma, mb = masks[DEFECTS[a]], masks[DEFECTS[b]]
            overlap = int(np.logical_and(ma, mb).sum())

            if overlap > 0:
                any_overlap = True

            if overlap > max_overlap_pixels:
                max_overlap_pixels = overlap
                max_overlap_details = (
                    image_id, DEFECTS[a], DEFECTS[b]
                )

            denom = int(ma.sum() + mb.sum())
            dice = (2 * overlap / denom) if denom else 0.0

            if dice > max_dice:
                max_dice = dice
                max_dice_details = (
                    image_id, DEFECTS[a], DEFECTS[b]
                )

print("Q6 max overlap pixels:", max_overlap_pixels)
print("Image and defect pair:", max_overlap_details)
print("Maximum pairwise Dice:", max_dice)
print("Pair for maximum Dice:", max_dice_details)
print("Spatially mutually exclusive:", "No" if any_overlap else "Yes")

Q6 max overlap pixels: 0
Image and defect pair: None
Maximum pairwise Dice: 0.0
Pair for maximum Dice: None
Spatially mutually exclusive: Yes


In [7]:
import torch
from torch.utils.data import DataLoader

print("Available DataLoaders:")
for name, obj in list(globals().items()):
    if isinstance(obj, DataLoader):
        print(name, "| batches:", len(obj), "| batch_size:", obj.batch_size)

Available DataLoaders:


In [8]:
print("Training rows:", len(train))
print("Unique images:", train["image_id"].nunique())

nonempty = (
    train["mask_rle"].notna()
    & train["mask_rle"].astype(str).str.strip().ne("")
)
print("Non-empty mask rows:", int(nonempty.sum()))

print("\nNon-empty masks by defect:")
print(train.loc[nonempty].groupby("defect_type")["image_id"].nunique())

Training rows: 45572
Unique images: 11393
Non-empty mask rows: 7201

Non-empty masks by defect:
defect_type
defect_a    1081
defect_b     313
defect_c    4100
defect_d    1707
Name: image_id, dtype: int64


In [6]:
import pandas as pd

print("Available history-like variables:")
for name, obj in list(globals().items()):
    if "history" in name.lower() or "log" in name.lower():
        if isinstance(obj, (list, dict, pd.DataFrame)):
            print(name, type(obj).__name__)

# If your training history is a list of dictionaries:
if "history" in globals():
    display(pd.DataFrame(history))

Available history-like variables:


In [9]:
import os
import glob

print("Possible training-history variables:")
for name, obj in list(globals().items()):
    if any(k in name.lower() for k in ["loss", "epoch", "history", "metrics", "logs"]):
        if isinstance(obj, (list, dict, tuple)):
            print(name, type(obj).__name__, str(obj)[:250])

print("\nPossible saved log files:")
for pattern in [
    "/kaggle/working/*.csv",
    "/kaggle/working/*.json",
    "/kaggle/working/*.txt"
]:
    for path in glob.glob(pattern):
        print(path)

Possible training-history variables:

Possible saved log files:


In [10]:
import torch
from torch.utils.data import DataLoader

print("Dataset objects:")
for name, obj in list(globals().items()):
    if hasattr(obj, "__len__") and hasattr(obj, "__getitem__"):
        if not isinstance(obj, (str, bytes, list, tuple, dict)):
            print(name, type(obj).__name__, "length:", len(obj))

Dataset objects:
train DataFrame length: 45572
image_rows DataFrame length: 4
rows DataFrame length: 4
presence DataFrame length: 11393
counts Series length: 4
cooccurrence DataFrame length: 4
co ndarray length: 4
correlation DataFrame length: 4
corr_values ndarray length: 4
match DataFrame length: 1
ma ndarray length: 128
mb ndarray length: 128
nonempty Series length: 45572


In [11]:
# Count non-empty masks by defect and inspect a few decoded masks
for defect in DEFECTS:
    subset = train[
        (train["defect_type"] == defect)
        & train["mask_rle"].notna()
        & train["mask_rle"].astype(str).str.strip().ne("")
    ]

    print(f"\n{defect}: {len(subset)} non-empty rows")

    if len(subset):
        row = subset.iloc[0]
        mask = decode_rle(row["mask_rle"])
        print("Sample image:", row["image_id"])
        print("Mask shape:", mask.shape)
        print("Positive pixels:", int(mask.sum()))


defect_a: 1081 non-empty rows
Sample image: 4S22I7SA34
Mask shape: (128, 800)
Positive pixels: 1100

defect_b: 313 non-empty rows
Sample image: JBAI4VEGD5
Mask shape: (128, 800)
Positive pixels: 306

defect_c: 4100 non-empty rows
Sample image: 30WEUKWBQT
Mask shape: (128, 800)
Positive pixels: 1750

defect_d: 1707 non-empty rows
Sample image: 6MBBS0WBUK
Mask shape: (128, 800)
Positive pixels: 17344


In [12]:
import inspect
import torch
from torch.utils.data import Dataset, DataLoader

print("=== Dataset classes ===")
for name, obj in list(globals().items()):
    try:
        if inspect.isclass(obj) and issubclass(obj, Dataset):
            if obj is not Dataset:
                print(name, inspect.signature(obj))
    except (TypeError, ValueError):
        pass

print("\n=== Image/path variables ===")
for name, obj in list(globals().items()):
    if any(k in name.lower() for k in ["image_dir", "img_dir", "train_dir", "data_dir", "transform"]):
        if isinstance(obj, (str, list, tuple, dict)):
            print(name, str(obj)[:250])

=== Dataset classes ===

=== Image/path variables ===


In [13]:
import glob
import os

print("Files in /kaggle/working:")
for path in glob.glob("/kaggle/working/**/*", recursive=True):
    if os.path.isfile(path):
        if path.endswith((".csv", ".json", ".txt", ".log", ".pth")):
            print(os.path.basename(path), os.path.getsize(path), "bytes")

Files in /kaggle/working:


In [14]:
import os
import glob
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")

print("\nDataset CSV files:")
for path in glob.glob("/kaggle/input/**/*.csv", recursive=True):
    if "train" in os.path.basename(path).lower():
        print(path)

print("\nDataset image directories:")
for path in glob.glob("/kaggle/input/**", recursive=True):
    if os.path.isdir(path) and any(
        word in os.path.basename(path).lower()
        for word in ["train_images", "images", "train"]
    ):
        print(path)

print("\nExisting model classes:")
for name in ["UNet", "SurfaceDataset", "train_loader", "val_loader"]:
    print(f"{name}: {name in globals()}")

GPU: Not available

Dataset CSV files:
/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv

Dataset image directories:
/kaggle/input/competitions/industrial-surface-defect-detection/public/train

Existing model classes:
UNet: False
SurfaceDataset: False
train_loader: False
val_loader: False


In [15]:
import os, glob, random
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE = "/kaggle/input/competitions/industrial-surface-defect-detection/public"
CSV_PATH = os.path.join(BASE, "train.csv")
IMAGE_DIR = os.path.join(BASE, "train")

df = pd.read_csv(CSV_PATH)
DEFECTS = ["defect_a", "defect_b", "defect_c", "defect_d"]
H, W = 128, 800

# Find images and index them by filename stem
image_paths = glob.glob(os.path.join(IMAGE_DIR, "**", "*"), recursive=True)
image_paths = [p for p in image_paths if os.path.isfile(p)]
image_map = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}

print("CSV shape:", df.shape)
print("Image files:", len(image_map))
print("Columns:", df.columns.tolist())
print("Example image IDs:", df["image_id"].head().tolist())

assert {"image_id", "defect_type", "mask_rle"}.issubset(df.columns)
assert len(image_map) > 0, "No images found. Check IMAGE_DIR."

# One row per image, four output mask channels
image_ids = sorted(df["image_id"].unique())
train_ids, val_ids = train_test_split(
    image_ids, test_size=0.2, random_state=SEED
)

def decode_rle(rle, height=H, width=W):
    mask = np.zeros(height * width, dtype=np.uint8)
    if pd.isna(rle) or not str(rle).strip():
        return mask.reshape((height, width), order="F")

    values = np.asarray(str(rle).split(), dtype=np.int64)
    starts = values[0::2] - 1
    lengths = values[1::2]

    for start, length in zip(starts, lengths):
        mask[start:start + length] = 1

    return mask.reshape((height, width), order="F")

class SurfaceDataset(Dataset):
    def __init__(self, ids):
        self.ids = list(ids)
        self.grouped = {
            image_id: {
                row["defect_type"]: row["mask_rle"]
                for _, row in group.iterrows()
            }
            for image_id, group in df[df["image_id"].isin(self.ids)]
                .groupby("image_id")
        }

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        path = image_map.get(image_id)

        if path is None:
            raise FileNotFoundError(f"Image not found: {image_id}")

        image = np.array(Image.open(path).convert("RGB"), dtype=np.float32) / 255.0
        image = torch.from_numpy(image).permute(2, 0, 1).float()

        rows = self.grouped[image_id]
        masks = np.stack([
            decode_rle(rows.get(defect, np.nan))
            for defect in DEFECTS
        ])
        masks = torch.from_numpy(masks.astype(np.float32))

        return image, masks

train_dataset = SurfaceDataset(train_ids)
val_dataset = SurfaceDataset(val_ids)

train_loader = DataLoader(
    train_dataset, batch_size=4, shuffle=True,
    num_workers=2, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_dataset, batch_size=4, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available()
)

images, masks = next(iter(train_loader))
print("Train images:", len(train_dataset), "| Validation images:", len(val_dataset))
print("Image shape/dtype:", images.shape, images.dtype)
print("Mask shape/dtype:", masks.shape, masks.dtype)

CSV shape: (45572, 5)
Image files: 11393
Columns: ['image_id', 'defect_type', 'mask_rle', 'width', 'height']
Example image IDs: ['4S22I7SA34', '4S22I7SA34', '4S22I7SA34', '4S22I7SA34', 'DZ899111OE']
Train images: 9114 | Validation images: 2279
Image shape/dtype: torch.Size([4, 3, 128, 800]) torch.float32
Mask shape/dtype: torch.Size([4, 4, 128, 800]) torch.float32


In [18]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPU abhi enabled nahi hai. Kaggle Settings > Accelerator > GPU select karo."
    )

CUDA available: False


RuntimeError: GPU abhi enabled nahi hai. Kaggle Settings > Accelerator > GPU select karo.

In [19]:
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=4, base=16):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        self.e1 = ConvBlock(in_channels, base)
        self.e2 = ConvBlock(base, base*2)
        self.e3 = ConvBlock(base*2, base*4)
        self.e4 = ConvBlock(base*4, base*8)
        self.b = ConvBlock(base*8, base*16)

        self.u4 = nn.ConvTranspose2d(base*16, base*8, 2, 2)
        self.d4 = ConvBlock(base*16, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.d3 = ConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2)
        self.d2 = ConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2)
        self.d1 = ConvBlock(base*2, base)

        self.out = nn.Conv2d(base, out_channels, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.b(self.pool(e4))

        d4 = self.d4(torch.cat([self.u4(b), e4], 1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))

        return self.out(d1)


model = UNet(in_channels=3, out_channels=4, base=16).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

print("Model ready:", next(model.parameters()).device)

Model ready: cpu


In [20]:
images, masks = next(iter(train_loader))

images = images.to(DEVICE)
masks = masks.to(DEVICE)

with torch.no_grad():
    logits = model(images)
    loss = criterion(logits, masks)

print("Images:", images.shape)
print("Masks:", masks.shape)
print("Predictions:", logits.shape)
print("Test loss:", round(loss.item(), 6))

assert logits.shape == masks.shape
print("Ready for 5-epoch training.")

Images: torch.Size([4, 3, 128, 800])
Masks: torch.Size([4, 4, 128, 800])
Predictions: torch.Size([4, 4, 128, 800])
Test loss: 0.778828
Ready for 5-epoch training.


In [21]:
import time
import torch

DEVICE = torch.device("cpu")
model = model.to(DEVICE)

model.train()
start = time.time()

for batch_idx, (images, masks) in enumerate(train_loader):
    if batch_idx >= 20:
        break

    images = images.to(DEVICE)
    masks = masks.to(DEVICE)

    optimizer.zero_grad(set_to_none=True)
    logits = model(images)
    loss = criterion(logits, masks)
    loss.backward()
    optimizer.step()

    print(f"Batch {batch_idx + 1}/20 | loss={loss.item():.4f}")

elapsed = time.time() - start
print(f"\n20 batches took {elapsed:.1f} seconds")
print(f"Estimated time per epoch: {elapsed * len(train_loader) / 20 / 60:.1f} minutes")

Batch 1/20 | loss=0.7826
Batch 2/20 | loss=0.7808
Batch 3/20 | loss=0.7675
Batch 4/20 | loss=0.7669
Batch 5/20 | loss=0.7714
Batch 6/20 | loss=0.7581
Batch 7/20 | loss=0.7708
Batch 8/20 | loss=0.7625
Batch 9/20 | loss=0.7493
Batch 10/20 | loss=0.7400
Batch 11/20 | loss=0.7461
Batch 12/20 | loss=0.7347
Batch 13/20 | loss=0.7394
Batch 14/20 | loss=0.7467
Batch 15/20 | loss=0.7404
Batch 16/20 | loss=0.7300
Batch 17/20 | loss=0.7278
Batch 18/20 | loss=0.7260
Batch 19/20 | loss=0.7400
Batch 20/20 | loss=0.7242

20 batches took 22.2 seconds
Estimated time per epoch: 42.2 minutes


In [22]:
import torch
import torch.nn as nn
import pandas as pd
import time

DEVICE = torch.device("cpu")

# Fresh initialization; pilot-trained weights are discarded
model = UNet(in_channels=3, out_channels=4, base=16).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

history = []
best_val_loss = float("inf")
best_epoch = None

print("Fresh baseline initialized on:", DEVICE)

Fresh baseline initialized on: cpu


In [ ]:
for epoch in range(1, 6):
    start = time.time()

    model.train()
    train_total = 0.0

    for images, masks in train_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        train_total += loss.item() * images.size(0)

    train_loss = train_total / len(train_dataset)

    model.eval()
    val_total = 0.0

    with torch.inference_mode():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, masks)
            val_total += loss.item() * images.size(0)

    val_loss = val_total / len(val_dataset)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(model.state_dict(),
                   "/kaggle/working/milestone2_baseline_best.pth")

    pd.DataFrame(history).to_csv(
        "/kaggle/working/milestone2_training_history.csv",
        index=False
    )

    print(
        f"Epoch {epoch}/5 | train={train_loss:.6f} | "
        f"val={val_loss:.6f} | "
        f"time={(time.time()-start)/60:.1f} min"
    )

In [ ]:
history_df = pd.DataFrame(history)
display(history_df)

first = history_df.iloc[0]
last = history_df.iloc[-1]

print(
    f"epoch1: ({first.train_loss:.6f}, {first.val_loss:.6f})"
)
print(
    f"epoch5: ({last.train_loss:.6f}, {last.val_loss:.6f})"
)
print("best_epoch:", int(history_df.loc[
    history_df["val_loss"].idxmin(), "epoch"
]))

if last.val_loss > history_df["val_loss"].min() and last.train_loss < first.train_loss:
    assessment = "possible overfitting after the best epoch"
elif last.train_loss < first.train_loss and last.val_loss < first.val_loss:
    assessment = "improving generalization over the five epochs"
else:
    assessment = "inconclusive; inspect both loss curves"

print("generalization_assessment:", assessment)